# Data balancing

The data is heavely unbalanced, if the nulls are deleted some classes are not even represented. In order to solve this some


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

import joblib
import optuna

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
dataset = pd.read_csv("thyroid0387_cleanded.data", sep = ";", index_col="Unnamed: 0")

In [13]:
dataset["diagnosis_group"].value_counts()

diagnosis_group
normal                   6547
hypothyroid               572
general_health            435
replacement_therapy       336
binding_protein           324
misc                      247
hyperthyroid              189
antithyroid_treatment      33
Name: count, dtype: int64

In [ ]:
dataset.dropna()["diagnosis_group"].value_counts()

diagnosis_group
normal                 15
hypothyroid             2
general_health          1
replacement_therapy     1
Name: count, dtype: int64

# Imputation of NaNs

In [15]:
dataset.dtypes

age                            int64
on_thyroxine                    bool
query_on_thyroxine              bool
on_antithyroid_medication       bool
sick                            bool
pregnant                        bool
thyroid_surgery                 bool
I131_treatment                  bool
query_hypothyroid               bool
query_hyperthyroid              bool
lithium                         bool
goitre                          bool
tumor                           bool
hypopituitary                   bool
psych                           bool
diagnosis                     object
diagnosis_group               object
sex_M                           bool
TSH_value                    float64
T3_value                     float64
TT4_value                    float64
T4U_value                    float64
FTI_value                    float64
TBG_value                    float64
ref_STMW                        bool
ref_SVHC                        bool
ref_SVHD                        bool
r

In [20]:
numeric_col = dataset.select_dtypes(include = ["int64", "float64"]).columns
categoric_col = dataset.select_dtypes(exclude = ["int64", "float64"]).columns

In [21]:
num_cols = dataset.select_dtypes(include=["float64", "int64"]).drop(["age"], axis=1).columns
cat_cols = dataset.select_dtypes(include=["bool"]).columns.tolist() + ["age"]

target = "diagnosis_group"
X = dataset.drop(columns=[target, "diagnosis"])
y = dataset[target]

In [22]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Imputació per numèriques amb mitjana
num_imputer = SimpleImputer(strategy="mean")

# Imputació per bools (categòriques) amb la moda (valor més freqüent)
cat_imputer = SimpleImputer(strategy="most_frequent")

# ColumnTransformer per aplicar imputació segons tipus
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_imputer, num_cols),
        ("cat", cat_imputer, cat_cols)
    ]
)


In [23]:
import pandas as pd

X_imputed = preprocessor.fit_transform(X)

# Tornar a DataFrame amb les columnes originals
X_imputed = pd.DataFrame(X_imputed, columns=num_cols.tolist() + cat_cols)


In [24]:
dataset.isnull().sum().sort_values(ascending=False)


TBG_value                    8370
T3_value                     2467
TSH_value                     787
T4U_value                     751
FTI_value                     744
TT4_value                     401
age                             0
on_thyroxine                    0
ref_WEST                        0
ref_SVI                         0
ref_SVHD                        0
ref_SVHC                        0
ref_STMW                        0
sex_M                           0
diagnosis_group                 0
diagnosis                       0
psych                           0
hypopituitary                   0
tumor                           0
goitre                          0
lithium                         0
query_hyperthyroid              0
query_hypothyroid               0
I131_treatment                  0
thyroid_surgery                 0
pregnant                        0
sick                            0
on_antithyroid_medication       0
query_on_thyroxine              0
ref_other     